# 08 · Main comparison on COCO (all methods, identical models)

**Purpose.** Every method works on the same detector candidates. Its operating point is selected on held-out (best-F1 and matched-recall), then evaluated unchanged on test, with image-level bootstrap 95 % CIs. Produces the main results table and the precision–recall frontier figure.

| | |
|---|---|
| **Proposal objective** | Objectives 2 and 3 |
| **Inputs** | all caches from notebooks 02, 03, 06, 07 |
| **Outputs** | `results/tables/main_comparison.{csv,md}`, `results/figures/*.png` |
| **Runtime (T4)** | ≈2–5 min (offline) |

> Run cells top to bottom. Every expensive stage is **resumable**: if Colab disconnects, reconnect,
> re-run from the top and it continues where it stopped (results are cached on Drive under `results/`).

In [ ]:
# ===================== Bootstrap (identical in every notebook) =====================
import os, sys, subprocess, shutil  # paths, sys.path, shell commands (git clone, pip), folder copy
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = (not IN_COLAB) and os.path.isdir("/kaggle/working")    # Kaggle notebook (needs Internet ON, see docs/KAGGLE_GUIDE.md)

# Where the project lives on Google Drive. Upload the whole repo folder there (see docs/COLAB_GUIDE.md),
# or set REPO_URL and it will be cloned into DRIVE_PROJECT on first run.
DRIVE_PROJECT = "/content/drive/MyDrive/detect-verify-decide"
REPO_URL = ""  # e.g. "https://github.com/<you>/detect-verify-decide.git"
BRANCH = "kaggle-version"  # git branch to take the latest code from (change it if you push to another branch)

if IN_COLAB:
    from google.colab import drive  # Colab only: mount Google Drive
    drive.mount("/content/drive")
    if not os.path.isdir(DRIVE_PROJECT) and REPO_URL:
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, DRIVE_PROJECT], check=True)
    elif os.path.isdir(os.path.join(DRIVE_PROJECT, ".git")):   # pick up the latest pushed code (never touches your notebooks/results)
        subprocess.run(["git", "-C", DRIVE_PROJECT, "fetch", "origin", BRANCH], check=False)
        subprocess.run(["git", "-C", DRIVE_PROJECT, "checkout", "FETCH_HEAD", "--", "src", "configs", "requirements.txt", "tools", "app"], check=False)
    PROJECT_ROOT = DRIVE_PROJECT
elif IN_KAGGLE:
    PROJECT_ROOT = "/kaggle/working/detect-verify-decide"
    if not os.path.isdir(os.path.join(PROJECT_ROOT, "src", "dvd")):
        import glob
        attached = glob.glob("/kaggle/input/*/src/dvd") + glob.glob("/kaggle/input/*/*/src/dvd")   # project attached as a Kaggle Dataset
        if attached:
            shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), PROJECT_ROOT)
        elif REPO_URL:
            url = REPO_URL
            try:   # private repo: add a Kaggle secret named GITHUB_TOKEN (Add-ons -> Secrets)
                from kaggle_secrets import UserSecretsClient
                url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
            except Exception:
                pass
            r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT_ROOT], capture_output=True)
            if r.returncode:   # (the message deliberately does not echo the command, which could contain the token)
                raise RuntimeError("git clone failed: is Internet ON, REPO_URL / BRANCH right, and for a private repo the GITHUB_TOKEN secret set?")
            subprocess.run(["git", "-C", PROJECT_ROOT, "remote", "set-url", "origin", REPO_URL])   # never keep the token in .git/config
        else:
            raise RuntimeError("Set REPO_URL (and BRANCH) above, or attach the project as a Kaggle Dataset.")
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

assert os.path.isdir(os.path.join(PROJECT_ROOT, "src", "dvd")), f"src/dvd not found under {PROJECT_ROOT}"
os.environ["DVD_ROOT"] = PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
# Install dependencies (Colab / Kaggle already have torch/torchvision with CUDA). ~1-2 min per fresh runtime.
if IN_COLAB or IN_KAGGLE:
    %pip install -q -r "{PROJECT_ROOT}/requirements.txt"

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt  # json files, numpy arrays, pandas tables, matplotlib plots
from dvd.config import load_config, save_selected, results_dir  # load configs/default.yaml, save tuned values, results folders
from dvd.io_utils import set_seed, save_json, load_json, JsonlCache  # seeding, JSON helpers and the resumable JSONL cache
from dvd import paths  # cache-file path helpers

cfg = load_config()
set_seed(cfg["seed"])
pd.set_option("display.precision", 4)
try:
    import torch  # PyTorch: GPU check and memory statistics
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only - expect very slow runs)")
except ImportError:
    print("torch not installed")
print("data_root   :", cfg["paths"]["data_root"])
print("results_root:", cfg["paths"]["results_root"])
print("tuned params:", cfg["selected"] or "(none yet)")

In [ ]:
from dvd.data.coco import prepare_annotations, fetch_images, CocoData, load_splits  # COCO annotations, image download, splits
prepare_annotations(cfg)   # annotation json only (the 241 MB zip is cached on Drive)
coco = CocoData(cfg)
splits = load_splits(cfg)
HELDOUT, TEST = splits["heldout"], splits["test"]
fetch_images(cfg, HELDOUT + TEST)   # only these images (a few hundred files), not the whole 778 MB val2017 zip
print(f"held-out: {len(HELDOUT)} images | test: {len(TEST)} images")

In [ ]:
from dvd.experiments import SplitData, available_methods, main_comparison, DET_ONLY, PROPOSED  # methods over identical candidates: sweeps, operating points, evaluation, costs
from dvd.viz import plot_pr  # plotting helpers (detections overlay, PR frontier, crop gallery)
H = SplitData(cfg, coco, "heldout", HELDOUT); T = SplitData(cfg, coco, "test", TEST)
MH, MT = available_methods(cfg, H), available_methods(cfg, T)
print("methods with complete caches:", list(MT))

In [ ]:
# selection on held-out, evaluation on test, bootstrap CIs + paired bootstrap against the detector-only baseline
main, sweeps_h, sweeps_t, base_h = main_comparison(cfg, H, T, MH, MT, n_boot=1000)
main[main.criterion == "best_f1"][["method", "precision", "recall", "f1", "fp_per_image", "fp_reduction", "fp_reduction_ci", "over_rejection_rate", "total_latency_s"]]

In [ ]:
main[main.criterion == "matched_recall"][["method", "precision", "recall", "d_recall_ci", "f1", "fp_per_image", "fp_reduction", "fp_reduction_ci", "over_rejection_rate", "total_latency_s"]]

## Save the tables

In [ ]:
tdir = results_dir(cfg, "tables")
main.to_csv(tdir / "main_comparison.csv", index=False)
fmt = main.copy()
for c in ["precision", "recall", "f1", "fp_reduction", "fp_removed_rate", "over_rejection_rate"]:
    fmt[c] = fmt[c].map(lambda x: f"{x:.3f}")
for c in ["f1_ci", "fp_reduction_ci", "d_recall_ci", "d_f1_ci"]:
    fmt[c] = fmt[c].map(lambda t: f"[{t[0]:.3f}, {t[1]:.3f}]")
cols = ["criterion", "method", "precision", "recall", "f1", "f1_ci", "fp_per_image", "fp_reduction", "fp_reduction_ci", "d_recall_ci", "over_rejection_rate", "total_latency_s", "total_invocations"]
(tdir / "main_comparison.md").write_text(fmt[cols].to_markdown(index=False, floatfmt=".3f"), encoding="utf-8")
print((tdir / "main_comparison.md").read_text())

## Precision–recall frontiers (test)

In [ ]:
sel = {n: main[(main.criterion == "best_f1") & (main.method == n)].iloc[0] for n in [DET_ONLY, PROPOSED] if n in MT}
plot_pr(sweeps_t, highlight={k: v for k, v in sel.items()}, title="All methods - test split")
plt.savefig(results_dir(cfg, "figures") / "pr_frontier_all_methods_test.png", dpi=150, bbox_inches="tight"); plt.show()

## Interpretation guide
- **Proposed vs Detector only** → does a verification stage reduce FPs at equal (matched) recall? (Objective 2)
- **Proposed vs Woodpecker-style** → same VLM, same candidates; difference = role separation + crop evidence. (Objective 3)
- **Proposed vs Global-view verifier + rule** → difference = crop-and-zoom evidence only.
- **Proposed vs Crop verifier decides alone** → difference = fusing detector confidence in a separate decision agent.
- **VCD / plain global yes/no** → a decoding-time remedy on global evidence.
- Overlapping bootstrap CIs mean the difference is not established on this test size; say so in the report.